# SmartPatches H5 visualizer

Interactive viewer for the `(raw, label, mask)` 3D patches inside an
H5 generated by `generate-training-data.py`.

Streams from disk via `h5py` (no full-array load); two sliders pick
the sample index **T** (across `N` patches) and the **Z** slice within
each `(Z, Y, X)` patch. Three subplots show the raw intensity, the
binary mask (if `mask` is present in the H5), and the integer label
image with a discrete colormap.

Edit `H5_PATH` below to point at your file.

In [1]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, IntSlider, Dropdown, fixed
from skimage.color import label2rgb

%matplotlib inline

H5_PATH = "/home/debian/jean-zay/segmentation_training/xenopus_segmentation.h5"

In [2]:
# Open lazily and inspect the layout (the file stays open for the
# notebook's lifetime so __getitem__ stays cheap).
h5 = h5py.File(H5_PATH, "r", swmr=True)

print(f"file: {H5_PATH}")
for split in ("train", "val"):
    if split not in h5:
        continue
    grp = h5[split]
    keys = list(grp.keys())
    raw_shape = grp["raw"].shape
    print(f"  /{split}: keys={keys}, raw shape={raw_shape}")

RuntimeError: Can't deserialize object header prefix (bad object header version number)

In [ ]:
def _show(split, t, z):
    grp = h5[split]
    raw = grp["raw"][t, z]                              # streamed slice
    label = grp["label"][t, z] if "label" in grp else None
    mask = grp["mask"][t, z] if "mask" in grp else None
    if mask is None and label is not None:
        mask = (label > 0).astype(np.uint8)              # derived on the fly
    n_panels = 1 + (mask is not None) + (label is not None)

    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
    axes = np.atleast_1d(axes)
    i = 0
    axes[i].imshow(raw, cmap="gray")
    axes[i].set_title(f"raw  T={t} Z={z}  (range {raw.min():.3g}–{raw.max():.3g})")
    axes[i].axis("off"); i += 1
    if mask is not None:
        axes[i].imshow(mask, cmap="gray", vmin=0, vmax=1)
        axes[i].set_title(f"binary mask  fg fraction = {mask.mean():.3f}")
        axes[i].axis("off"); i += 1
    if label is not None:
        n_objects = len(np.unique(label)) - (1 if 0 in np.unique(label) else 0)
        axes[i].imshow(label2rgb(label, bg_label=0))
        axes[i].set_title(f"integer labels  ({n_objects} objects)")
        axes[i].axis("off"); i += 1
    plt.tight_layout(); plt.show()


def viewer(split="train"):
    grp = h5[split]
    n_samples = grp["raw"].shape[0]
    n_z = grp["raw"].shape[1] if grp["raw"].ndim == 4 else 1
    if n_samples == 0:
        print(f"/{split} is empty")
        return
    interact(
        _show,
        split=fixed(split),
        t=IntSlider(min=0, max=n_samples - 1, step=1, value=0, description="T (sample)"),
        z=IntSlider(min=0, max=max(0, n_z - 1), step=1, value=n_z // 2, description="Z (slice)"),
    )

## Train split

In [ ]:
viewer("train")

## Val split

In [ ]:
viewer("val")

## Summary statistics across the dataset

In [ ]:
for split in ("train", "val"):
    if split not in h5:
        continue
    grp = h5[split]
    n = grp["raw"].shape[0]
    if n == 0:
        continue
    if "label" in grp:
        # Sample 100 patches at most to keep this cheap.
        idx = np.linspace(0, n - 1, num=min(100, n), dtype=int)
        fg_fracs = []
        n_objects = []
        for i in idx:
            lab = grp["label"][i]
            fg_fracs.append(float((lab > 0).mean()))
            n_objects.append(int(len(np.unique(lab)) - (1 if 0 in np.unique(lab) else 0)))
        print(f"/{split}  ({n} patches, sampled {len(idx)})")
        print(f"  fg fraction: min={min(fg_fracs):.3f}  median={np.median(fg_fracs):.3f}  max={max(fg_fracs):.3f}")
        print(f"  objects/patch: min={min(n_objects)}  median={int(np.median(n_objects))}  max={max(n_objects)}")

In [ ]:
# Close the file when done.
h5.close()